# 02 - Modelagem e Simulacao Computacional
> Construcao de modelos fisicos, validacao de equacoes e condicoes de contorno

**Modulo:** `11_engfis_tecnico` | **Feito com:** [Claude](https://claude.ai) (Anthropic)

---


## Setup e Importacoes

Configuracao do ambiente com as bibliotecas necessarias para modelagem e simulacao.

In [ ]:
import os
from anthropic import Anthropic
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from IPython.display import display, Markdown

# Configuracao do cliente Claude
client = Anthropic()  # usa ANTHROPIC_API_KEY do ambiente
MODEL = "claude-sonnet-4-20250514"

# Configuracao de plots
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True

def ask_claude(prompt, system="Voce e um fisico computacional especialista. Responda em portugues sem acentos."):
    """Funcao auxiliar para consultar Claude."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=4096,
        system=system,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text

print("Ambiente configurado com sucesso!")

---

## Construindo Modelos Fisicos com Claude

Vamos pedir ao Claude para formular as equacoes diferenciais que governam a conducao
de calor em uma barra unidimensional, incluindo condicoes de contorno e iniciais.

In [ ]:
# Pedir ao Claude para formular o modelo fisico
prompt_modelo = """
Formule o modelo matematico completo para conducao de calor em uma barra
unidimensional de comprimento L = 1m, feita de cobre.

Inclua:
1. A equacao diferencial parcial governante (equacao do calor 1D)
2. Condicoes de contorno: temperatura fixa T=100C em x=0, isolado em x=L
3. Condicao inicial: T(x,0) = 20C (temperatura ambiente)
4. Valores tipicos dos parametros fisicos do cobre
5. A forma adimensional da equacao

Apresente de forma clara e organizada.
"""

resposta_modelo = ask_claude(prompt_modelo)
display(Markdown(resposta_modelo))

In [ ]:
# Implementar a solucao numerica do modelo de conducao de calor 1D
# Metodo: Diferencas Finitas Explicito (FTCS - Forward Time Central Space)

# Parametros fisicos do cobre
L = 1.0          # comprimento da barra [m]
alpha = 1.11e-4   # difusividade termica do cobre [m^2/s]
T_left = 100.0    # temperatura na extremidade esquerda [C]
T_init = 20.0     # temperatura inicial [C]

# Parametros numericos
Nx = 50           # numero de pontos espaciais
dx = L / (Nx - 1)
dt = 0.4 * dx**2 / alpha  # criterio de estabilidade: dt < dx^2 / (2*alpha)
t_final = 500.0   # tempo final de simulacao [s]
Nt = int(t_final / dt)

print(f"dx = {dx:.4f} m")
print(f"dt = {dt:.4f} s")
print(f"Numero de passos temporais: {Nt}")
print(f"Numero de Fourier (r = alpha*dt/dx^2): {alpha*dt/dx**2:.4f}")

# Malha espacial
x = np.linspace(0, L, Nx)

# Condicao inicial
T = np.full(Nx, T_init)
T[0] = T_left  # CC: temperatura fixa em x=0

# Armazenar snapshots para visualizacao
snapshots = [T.copy()]
tempos_snap = [0.0]
intervalo_snap = Nt // 8

# Loop temporal (FTCS)
for n in range(1, Nt + 1):
    T_new = T.copy()
    r = alpha * dt / dx**2
    # Pontos internos
    for i in range(1, Nx - 1):
        T_new[i] = T[i] + r * (T[i+1] - 2*T[i] + T[i-1])
    # CC: Neumann em x=L (isolado: dT/dx = 0)
    T_new[-1] = T_new[-2]
    # CC: Dirichlet em x=0
    T_new[0] = T_left
    T = T_new
    
    if n % intervalo_snap == 0:
        snapshots.append(T.copy())
        tempos_snap.append(n * dt)

# Visualizacao
fig, ax = plt.subplots(figsize=(10, 6))
colors = cm.hot(np.linspace(0.2, 0.9, len(snapshots)))

for i, (T_snap, t) in enumerate(zip(snapshots, tempos_snap)):
    ax.plot(x, T_snap, color=colors[i], linewidth=2, label=f't = {t:.1f} s')

ax.set_xlabel('Posicao x [m]')
ax.set_ylabel('Temperatura [C]')
ax.set_title('Conducao de Calor 1D em Barra de Cobre (FTCS)')
ax.legend(loc='lower right', fontsize=9)
ax.set_ylim(15, 105)
plt.tight_layout()
plt.show()

---

## Validacao de Equacoes Diferenciais

Apresentamos uma EDP ao Claude para verificar a consistencia fisica,
analise dimensional e se as condicoes de contorno estao bem postas.

In [ ]:
# Pedir ao Claude para validar uma equacao diferencial
prompt_validacao = """
Analise criticamente o seguinte modelo de EDP para difusao de calor 2D
em uma placa quadrada:

Equacao: dT/dt = k * (d2T/dx2 + d2T/dy2) + Q/(rho*cp)

onde:
- T(x,y,t) e a temperatura [K]
- k = 237 W/(m*K) (condutividade termica do aluminio)
- rho = 2700 kg/m^3 (densidade)
- cp = 900 J/(kg*K) (calor especifico)
- Q = 1000 W/m^3 (fonte de calor volumetrica)

Condicoes de contorno:
- T(0,y,t) = 300 K
- T(L,y,t) = 300 K  
- dT/dy(x,0,t) = 0 (isolado)
- T(x,L,t) = 400 K

Condicao inicial: T(x,y,0) = 300 K

Verifique:
1. A consistencia dimensional de cada termo
2. Se o coeficiente correto e alpha=k/(rho*cp) e nao apenas k
3. Se as condicoes de contorno sao fisicamente razoaveis
4. Se o problema esta bem posto (existencia e unicidade)
5. Sugira correcoes se necessario
"""

resposta_validacao = ask_claude(prompt_validacao)
display(Markdown(resposta_validacao))

In [ ]:
# Demonstrar a analise dimensional programaticamente
print("=== Analise Dimensional ===")
print()

# Parametros
k = 237      # W/(m*K) = kg*m/(s^3*K)
rho = 2700   # kg/m^3
cp = 900     # J/(kg*K) = m^2/(s^2*K)
Q = 1000     # W/m^3 = kg/(m*s^3)

alpha_calc = k / (rho * cp)
print(f"Difusividade termica alpha = k/(rho*cp) = {alpha_calc:.6e} m^2/s")
print()

# Verificacao dimensional
print("Termo dT/dt:")
print("  [K/s]")
print()
print("Termo k*(d2T/dx2) [INCORRETO]:")
print("  [W/(m*K)] * [K/m^2] = [W/m^3] = [kg/(m*s^3)]")
print("  NAO e [K/s] -> INCONSISTENTE!")
print()
print("Termo alpha*(d2T/dx2) [CORRETO]:")
print("  [m^2/s] * [K/m^2] = [K/s]")
print("  E [K/s] -> CONSISTENTE!")
print()
print("Termo Q/(rho*cp):")
print(f"  [W/m^3] / ([kg/m^3]*[J/(kg*K)]) = [K/s]")
print(f"  Valor = {Q/(rho*cp):.6e} K/s")
print()
print("Equacao corrigida:")
print("  dT/dt = alpha*(d2T/dx2 + d2T/dy2) + Q/(rho*cp)")

---

## Simulacao de Dinamica Molecular Simplificada

Vamos pedir ao Claude para gerar codigo de uma simulacao de dinamica molecular
simplificada com potencial de Lennard-Jones em 2D.

In [ ]:
# Pedir ao Claude a fundamentacao teorica
prompt_md = """
Explique brevemente o potencial de Lennard-Jones e como ele e usado
em simulacoes de dinamica molecular. Inclua:
1. A forma funcional V(r) = 4*epsilon*[(sigma/r)^12 - (sigma/r)^6]
2. Significado fisico dos termos repulsivo e atrativo
3. Como calcular forcas a partir do potencial
4. O algoritmo Velocity Verlet para integracao temporal
"""

resposta_md = ask_claude(prompt_md)
display(Markdown(resposta_md))

In [ ]:
# Simulacao de Dinamica Molecular 2D com Lennard-Jones

# Parametros LJ (unidades reduzidas)
epsilon = 1.0
sigma = 1.0
r_cut = 2.5 * sigma  # raio de corte
dt_md = 0.005         # passo de tempo
n_particles = 16      # numero de particulas
box_size = 8.0        # tamanho da caixa
n_steps_md = 2000     # passos de simulacao

def lj_force(r):
    """Calcula forca de Lennard-Jones (magnitude) para distancia r."""
    if r < 0.8 * sigma:
        r = 0.8 * sigma  # evitar singularidade
    sr6 = (sigma / r)**6
    sr12 = sr6**2
    return 24 * epsilon * (2 * sr12 - sr6) / r

def lj_potential(r):
    """Calcula potencial de Lennard-Jones."""
    sr6 = (sigma / r)**6
    return 4 * epsilon * (sr6**2 - sr6)

def compute_forces(pos, box):
    """Calcula forcas em todas as particulas com condicoes periodicas."""
    n = len(pos)
    forces = np.zeros_like(pos)
    pe = 0.0
    
    for i in range(n):
        for j in range(i + 1, n):
            dr = pos[j] - pos[i]
            # Condicoes periodicas (imagem minima)
            dr -= box * np.round(dr / box)
            r = np.linalg.norm(dr)
            
            if r < r_cut:
                f_mag = lj_force(r)
                f_vec = f_mag * dr / r
                forces[i] += f_vec
                forces[j] -= f_vec
                pe += lj_potential(r)
    
    return forces, pe

# Inicializacao: grade regular com pequena perturbacao
n_side = int(np.sqrt(n_particles))
spacing = box_size / (n_side + 1)
positions = np.zeros((n_particles, 2))
for i in range(n_side):
    for j in range(n_side):
        idx = i * n_side + j
        positions[idx] = [(i + 1) * spacing, (j + 1) * spacing]

# Velocidades iniciais aleatorias (distribuicao Maxwell-Boltzmann simplificada)
np.random.seed(42)
T_target = 1.0  # temperatura alvo
velocities = np.random.randn(n_particles, 2) * np.sqrt(T_target)
velocities -= velocities.mean(axis=0)  # remover momento total

print(f"Simulacao MD 2D: {n_particles} particulas em caixa {box_size}x{box_size}")
print(f"Passo de tempo: {dt_md}, Passos totais: {n_steps_md}")
print(f"Temperatura alvo: {T_target} (unidades reduzidas)")

In [ ]:
# Executar simulacao MD com Velocity Verlet
trajectories = [positions.copy()]
energias_cin = []
energias_pot = []

forces, pe = compute_forces(positions, box_size)

for step in range(n_steps_md):
    # Velocity Verlet - passo 1: atualizar posicoes
    positions = positions + velocities * dt_md + 0.5 * forces * dt_md**2
    # Condicoes periodicas
    positions = positions % box_size
    
    # Velocity Verlet - passo 2: calcular novas forcas
    forces_new, pe = compute_forces(positions, box_size)
    
    # Velocity Verlet - passo 3: atualizar velocidades
    velocities = velocities + 0.5 * (forces + forces_new) * dt_md
    forces = forces_new
    
    # Energia cinetica
    ke = 0.5 * np.sum(velocities**2)
    energias_cin.append(ke)
    energias_pot.append(pe)
    
    # Salvar snapshots
    if step % 20 == 0:
        trajectories.append(positions.copy())

print(f"Simulacao concluida! {len(trajectories)} snapshots salvos.")
print(f"Energia cinetica media: {np.mean(energias_cin):.3f}")
print(f"Energia potencial media: {np.mean(energias_pot):.3f}")
print(f"Energia total media: {np.mean(np.array(energias_cin) + np.array(energias_pot)):.3f}")

In [ ]:
# Visualizacao das trajetorias e energia
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Configuracao inicial e final
ax1 = axes[0]
ax1.scatter(trajectories[0][:, 0], trajectories[0][:, 1],
            c='blue', s=100, alpha=0.6, label='Inicial', zorder=5)
ax1.scatter(trajectories[-1][:, 0], trajectories[-1][:, 1],
            c='red', s=100, alpha=0.6, label='Final', zorder=5)
ax1.set_xlim(0, box_size)
ax1.set_ylim(0, box_size)
ax1.set_aspect('equal')
ax1.set_title('Posicoes Inicial e Final')
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.legend()

# Plot 2: Trajetorias de algumas particulas
ax2 = axes[1]
traj_array = np.array(trajectories)
colors_p = plt.cm.tab10(np.linspace(0, 1, min(6, n_particles)))
for p in range(min(6, n_particles)):
    ax2.plot(traj_array[:, p, 0], traj_array[:, p, 1],
             '-', color=colors_p[p], alpha=0.5, linewidth=0.8)
    ax2.scatter(traj_array[-1, p, 0], traj_array[-1, p, 1],
               c=[colors_p[p]], s=60, zorder=5)
ax2.set_xlim(0, box_size)
ax2.set_ylim(0, box_size)
ax2.set_aspect('equal')
ax2.set_title('Trajetorias (6 particulas)')
ax2.set_xlabel('x')
ax2.set_ylabel('y')

# Plot 3: Evolucao da energia
ax3 = axes[2]
passos = np.arange(len(energias_cin))
e_total = np.array(energias_cin) + np.array(energias_pot)
ax3.plot(passos, energias_cin, 'r-', alpha=0.5, label='Cinetica')
ax3.plot(passos, energias_pot, 'b-', alpha=0.5, label='Potencial')
ax3.plot(passos, e_total, 'k-', linewidth=2, label='Total')
ax3.set_xlabel('Passo')
ax3.set_ylabel('Energia (un. reduzidas)')
ax3.set_title('Conservacao de Energia')
ax3.legend()

plt.tight_layout()
plt.show()

---

## Metodo de Diferencas Finitas

Vamos pedir ao Claude para implementar um solver da equacao de Laplace 2D
usando diferencas finitas (metodo iterativo de Gauss-Seidel).

In [ ]:
# Pedir ao Claude para explicar a abordagem
prompt_laplace = """
Explique como resolver a equacao de Laplace 2D (nabla^2 T = 0) usando
o metodo de diferencas finitas com iteracao de Gauss-Seidel.

Inclua:
1. A discretizacao da equacao em uma grade retangular
2. O esquema iterativo de Gauss-Seidel
3. Criterio de convergencia
4. Por que Gauss-Seidel converge mais rapido que Jacobi
"""

resposta_laplace = ask_claude(prompt_laplace)
display(Markdown(resposta_laplace))

In [ ]:
# Solver da equacao de Laplace 2D - Metodo de Gauss-Seidel

# Parametros da grade
Nx_lap = 50
Ny_lap = 50
Lx = 1.0
Ly = 1.0
dx_lap = Lx / (Nx_lap - 1)
dy_lap = Ly / (Ny_lap - 1)

# Inicializacao
T_lap = np.zeros((Ny_lap, Nx_lap))

# Condicoes de contorno
T_lap[0, :] = 0.0      # inferior: T = 0
T_lap[-1, :] = 100.0   # superior: T = 100
T_lap[:, 0] = 0.0      # esquerda: T = 0
T_lap[:, -1] = 0.0     # direita: T = 0

# Iteracao de Gauss-Seidel
max_iter = 10000
tol = 1e-6
historico_erro = []

for it in range(max_iter):
    max_diff = 0.0
    for j in range(1, Ny_lap - 1):
        for i in range(1, Nx_lap - 1):
            T_old = T_lap[j, i]
            T_lap[j, i] = 0.25 * (T_lap[j, i+1] + T_lap[j, i-1] +
                                    T_lap[j+1, i] + T_lap[j-1, i])
            diff = abs(T_lap[j, i] - T_old)
            if diff > max_diff:
                max_diff = diff
    
    historico_erro.append(max_diff)
    
    if max_diff < tol:
        print(f"Convergiu em {it + 1} iteracoes (erro = {max_diff:.2e})")
        break
else:
    print(f"Nao convergiu apos {max_iter} iteracoes (erro = {max_diff:.2e})")

# Malhas para visualizacao
x_lap = np.linspace(0, Lx, Nx_lap)
y_lap = np.linspace(0, Ly, Ny_lap)
X_lap, Y_lap = np.meshgrid(x_lap, y_lap)

In [ ]:
# Visualizacao da solucao da equacao de Laplace
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Mapa de contorno preenchido
ax1 = axes[0]
cf = ax1.contourf(X_lap, Y_lap, T_lap, levels=20, cmap='hot')
plt.colorbar(cf, ax=ax1, label='Temperatura')
ax1.set_title('Equacao de Laplace 2D - Contorno')
ax1.set_xlabel('x [m]')
ax1.set_ylabel('y [m]')
ax1.set_aspect('equal')

# Plot 2: Linhas de contorno
ax2 = axes[1]
cs = ax2.contour(X_lap, Y_lap, T_lap, levels=15, cmap='hot')
ax2.clabel(cs, inline=True, fontsize=8, fmt='%.0f')
ax2.set_title('Isotermas')
ax2.set_xlabel('x [m]')
ax2.set_ylabel('y [m]')
ax2.set_aspect('equal')

# Plot 3: Convergencia
ax3 = axes[2]
ax3.semilogy(historico_erro, 'b-', linewidth=1)
ax3.axhline(y=tol, color='r', linestyle='--', label=f'Tolerancia = {tol}')
ax3.set_xlabel('Iteracao')
ax3.set_ylabel('Erro maximo')
ax3.set_title('Convergencia de Gauss-Seidel')
ax3.legend()

plt.tight_layout()
plt.show()

---

## Simulacao de Fenomenos Ondulatorios

Simulacao da equacao de onda 1D para uma corda vibrante usando diferencas finitas.

In [ ]:
# Pedir ao Claude a teoria
prompt_onda = """
Descreva a equacao de onda 1D para uma corda vibrante:
d2u/dt2 = c^2 * d2u/dx2

Inclua:
1. Significado fisico de cada termo
2. Condicoes de contorno para uma corda fixa nas extremidades
3. Como implementar numericamente com diferencas finitas (esquema explicito)
4. O criterio de estabilidade CFL: c*dt/dx <= 1
5. Solucao analitica por modos normais
"""

resposta_onda = ask_claude(prompt_onda)
display(Markdown(resposta_onda))

In [ ]:
# Simulacao da equacao de onda 1D - corda vibrante

# Parametros
L_onda = 1.0        # comprimento da corda [m]
c_onda = 1.0        # velocidade de propagacao [m/s]
Nx_onda = 200       # pontos espaciais
dx_onda = L_onda / (Nx_onda - 1)
dt_onda = 0.8 * dx_onda / c_onda  # CFL < 1
t_final_onda = 3.0  # tempo final
Nt_onda = int(t_final_onda / dt_onda)

courant = c_onda * dt_onda / dx_onda
print(f"Numero de Courant: {courant:.4f}")
print(f"dx = {dx_onda:.5f}, dt = {dt_onda:.5f}")
print(f"Passos temporais: {Nt_onda}")

x_onda = np.linspace(0, L_onda, Nx_onda)

# Condicao inicial: combinacao de dois modos normais
u_prev = np.sin(np.pi * x_onda / L_onda) + 0.3 * np.sin(3 * np.pi * x_onda / L_onda)
u_prev[0] = 0.0  # CC: extremidades fixas
u_prev[-1] = 0.0

# Velocidade inicial: zero
# Primeiro passo usando expansao de Taylor
r2 = courant**2
u_curr = np.zeros(Nx_onda)
for i in range(1, Nx_onda - 1):
    u_curr[i] = u_prev[i] + 0.5 * r2 * (u_prev[i+1] - 2*u_prev[i] + u_prev[i-1])

# Armazenar snapshots
snapshots_onda = [(0.0, u_prev.copy())]
snap_times = [0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 2.5]
snap_idx = 1

# Loop temporal
for n in range(1, Nt_onda):
    u_next = np.zeros(Nx_onda)
    for i in range(1, Nx_onda - 1):
        u_next[i] = (2 * u_curr[i] - u_prev[i] +
                      r2 * (u_curr[i+1] - 2*u_curr[i] + u_curr[i-1]))
    # CC: extremidades fixas
    u_next[0] = 0.0
    u_next[-1] = 0.0
    
    t_now = (n + 1) * dt_onda
    
    # Salvar snapshot
    if snap_idx < len(snap_times) and t_now >= snap_times[snap_idx]:
        snapshots_onda.append((t_now, u_next.copy()))
        snap_idx += 1
    
    u_prev = u_curr.copy()
    u_curr = u_next.copy()

print(f"Simulacao concluida! {len(snapshots_onda)} snapshots capturados.")

In [ ]:
# Visualizacao: snapshots da corda vibrante
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes_flat = axes.flatten()

for idx, (t, u) in enumerate(snapshots_onda[:8]):
    ax = axes_flat[idx]
    ax.fill_between(x_onda, 0, u, alpha=0.3, color='steelblue')
    ax.plot(x_onda, u, 'b-', linewidth=2)
    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.set_title(f't = {t:.2f} s')
    ax.set_xlim(0, L_onda)
    ax.set_ylim(-1.5, 1.5)
    ax.set_xlabel('x [m]')
    ax.set_ylabel('u(x,t)')

fig.suptitle('Equacao de Onda 1D - Corda Vibrante (Snapshots Temporais)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Validacao: Comparando com Solucoes Analiticas

Para o oscilador harmonico simples, comparamos a solucao numerica
(Euler, RK4) com a solucao analitica exata para quantificar erros.

In [ ]:
# Pedir ao Claude para analisar os metodos
prompt_validacao_osc = """
Compare os metodos de Euler explicito e Runge-Kutta de 4a ordem (RK4)
para resolver o oscilador harmonico simples:

d2x/dt2 + omega^2 * x = 0

Discuta:
1. A solucao analitica exata
2. Por que o metodo de Euler diverge (nao conserva energia)
3. Vantagens do RK4 em termos de ordem de convergencia
4. Como quantificar o erro numerico
"""

resposta_osc = ask_claude(prompt_validacao_osc)
display(Markdown(resposta_osc))

In [ ]:
# Oscilador Harmonico: Euler vs RK4 vs Analitico

omega = 2 * np.pi  # frequencia angular [rad/s]
x0 = 1.0           # posicao inicial
v0 = 0.0           # velocidade inicial
t_final_osc = 5.0  # tempo final [s]
dt_osc = 0.01      # passo de tempo
t_arr = np.arange(0, t_final_osc, dt_osc)
N_osc = len(t_arr)

# Solucao analitica
x_analitico = x0 * np.cos(omega * t_arr) + (v0 / omega) * np.sin(omega * t_arr)
v_analitico = -x0 * omega * np.sin(omega * t_arr) + v0 * np.cos(omega * t_arr)

# ---- Metodo de Euler Explicito ----
x_euler = np.zeros(N_osc)
v_euler = np.zeros(N_osc)
x_euler[0] = x0
v_euler[0] = v0

for i in range(N_osc - 1):
    x_euler[i+1] = x_euler[i] + dt_osc * v_euler[i]
    v_euler[i+1] = v_euler[i] - dt_osc * omega**2 * x_euler[i]

# ---- Metodo RK4 ----
def f_osc(state, t):
    """Derivadas do oscilador: state = [x, v]"""
    x, v = state
    return np.array([v, -omega**2 * x])

x_rk4 = np.zeros(N_osc)
v_rk4 = np.zeros(N_osc)
x_rk4[0] = x0
v_rk4[0] = v0
state = np.array([x0, v0])

for i in range(N_osc - 1):
    t = t_arr[i]
    k1 = f_osc(state, t)
    k2 = f_osc(state + 0.5 * dt_osc * k1, t + 0.5 * dt_osc)
    k3 = f_osc(state + 0.5 * dt_osc * k2, t + 0.5 * dt_osc)
    k4 = f_osc(state + dt_osc * k3, t + dt_osc)
    state = state + (dt_osc / 6.0) * (k1 + 2*k2 + 2*k3 + k4)
    x_rk4[i+1] = state[0]
    v_rk4[i+1] = state[1]

# Erros
erro_euler = np.abs(x_euler - x_analitico)
erro_rk4 = np.abs(x_rk4 - x_analitico)

print(f"Erro maximo Euler: {np.max(erro_euler):.6f}")
print(f"Erro maximo RK4:   {np.max(erro_rk4):.2e}")
print(f"Razao: Euler/RK4 = {np.max(erro_euler)/np.max(erro_rk4):.0f}x")

In [ ]:
# Visualizacao completa da comparacao
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Posicao vs tempo
ax1 = axes[0, 0]
ax1.plot(t_arr, x_analitico, 'k-', linewidth=2, label='Analitico')
ax1.plot(t_arr, x_euler, 'r--', linewidth=1.5, alpha=0.7, label='Euler')
ax1.plot(t_arr, x_rk4, 'b:', linewidth=2, alpha=0.7, label='RK4')
ax1.set_xlabel('Tempo [s]')
ax1.set_ylabel('Posicao x(t)')
ax1.set_title('Oscilador Harmonico: x(t)')
ax1.legend()

# Plot 2: Espaco de fase
ax2 = axes[0, 1]
ax2.plot(x_analitico, v_analitico, 'k-', linewidth=2, label='Analitico')
ax2.plot(x_euler, v_euler, 'r-', linewidth=1, alpha=0.5, label='Euler')
ax2.plot(x_rk4, v_rk4, 'b--', linewidth=1.5, alpha=0.7, label='RK4')
ax2.set_xlabel('Posicao x')
ax2.set_ylabel('Velocidade v')
ax2.set_title('Espaco de Fase')
ax2.set_aspect('equal')
ax2.legend()

# Plot 3: Erro absoluto
ax3 = axes[1, 0]
ax3.semilogy(t_arr, erro_euler, 'r-', linewidth=1.5, label='Euler')
ax3.semilogy(t_arr, erro_rk4 + 1e-16, 'b-', linewidth=1.5, label='RK4')
ax3.set_xlabel('Tempo [s]')
ax3.set_ylabel('Erro absoluto |x_num - x_anal|')
ax3.set_title('Evolucao do Erro')
ax3.legend()

# Plot 4: Energia total
ax4 = axes[1, 1]
E_analitico = 0.5 * v_analitico**2 + 0.5 * omega**2 * x_analitico**2
E_euler = 0.5 * v_euler**2 + 0.5 * omega**2 * x_euler**2
E_rk4 = 0.5 * v_rk4**2 + 0.5 * omega**2 * x_rk4**2
ax4.plot(t_arr, E_analitico, 'k-', linewidth=2, label='Analitico')
ax4.plot(t_arr, E_euler, 'r-', linewidth=1.5, alpha=0.7, label='Euler')
ax4.plot(t_arr, E_rk4, 'b-', linewidth=1.5, alpha=0.7, label='RK4')
ax4.set_xlabel('Tempo [s]')
ax4.set_ylabel('Energia Total')
ax4.set_title('Conservacao de Energia')
ax4.legend()

fig.suptitle('Validacao: Oscilador Harmonico - Numerico vs Analitico',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Analise de convergencia: erro vs passo de tempo
dts_teste = [0.1, 0.05, 0.02, 0.01, 0.005, 0.002, 0.001]
erros_euler_conv = []
erros_rk4_conv = []

for dt_teste in dts_teste:
    t_test = np.arange(0, 2.0, dt_teste)  # simular 2 segundos
    n_test = len(t_test)
    
    # Analitico
    x_anal_test = x0 * np.cos(omega * t_test)
    
    # Euler
    xe = np.zeros(n_test)
    ve = np.zeros(n_test)
    xe[0], ve[0] = x0, v0
    for i in range(n_test - 1):
        xe[i+1] = xe[i] + dt_teste * ve[i]
        ve[i+1] = ve[i] - dt_teste * omega**2 * xe[i]
    erros_euler_conv.append(np.max(np.abs(xe - x_anal_test)))
    
    # RK4
    xr = np.zeros(n_test)
    vr = np.zeros(n_test)
    xr[0], vr[0] = x0, v0
    st = np.array([x0, v0])
    for i in range(n_test - 1):
        t = t_test[i]
        k1 = f_osc(st, t)
        k2 = f_osc(st + 0.5*dt_teste*k1, t + 0.5*dt_teste)
        k3 = f_osc(st + 0.5*dt_teste*k2, t + 0.5*dt_teste)
        k4 = f_osc(st + dt_teste*k3, t + dt_teste)
        st = st + (dt_teste/6.0)*(k1 + 2*k2 + 2*k3 + k4)
        xr[i+1] = st[0]
        vr[i+1] = st[1]
    erros_rk4_conv.append(np.max(np.abs(xr - x_anal_test)))

# Plot de convergencia
fig, ax = plt.subplots(figsize=(8, 6))
ax.loglog(dts_teste, erros_euler_conv, 'ro-', linewidth=2, markersize=8, label='Euler (esperado: O(dt))')
ax.loglog(dts_teste, erros_rk4_conv, 'bs-', linewidth=2, markersize=8, label='RK4 (esperado: O(dt^4))')

# Linhas de referencia
dts_ref = np.array(dts_teste)
ax.loglog(dts_ref, 5 * dts_ref, 'r--', alpha=0.4, label='O(dt)')
ax.loglog(dts_ref, 500 * dts_ref**4, 'b--', alpha=0.4, label='O(dt^4)')

ax.set_xlabel('Passo de tempo dt [s]')
ax.set_ylabel('Erro maximo')
ax.set_title('Analise de Convergencia: Euler vs RK4')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

---

## Exercicios Propostos

### Exercicio 1: Conducao com Fonte de Calor
Modifique a simulacao de conducao 1D para incluir uma fonte de calor
interna Q(x) = Q0 * sin(pi*x/L). Use Claude para formular a equacao
modificada e implemente a solucao numerica.

### Exercicio 2: Potencial de Morse
Substitua o potencial de Lennard-Jones pelo potencial de Morse na
simulacao MD. Peca ao Claude para explicar as diferencas fisicas
e implemente a nova funcao de forca.

### Exercicio 3: Equacao de Poisson
Modifique o solver de Laplace para resolver a equacao de Poisson
(nabla^2 T = -f(x,y)) com uma fonte termica gaussiana centrada
no dominio. Visualize o efeito da fonte.

### Exercicio 4: Corda com Amortecimento
Adicione um termo de amortecimento a equacao de onda:
d2u/dt2 + gamma*du/dt = c^2 * d2u/dx2
Peca ao Claude para verificar a estabilidade numerica.

### Exercicio 5: Metodo Simpletico
Implemente o metodo Stormer-Verlet (simpletico) para o oscilador
harmonico e compare a conservacao de energia com Euler e RK4
em simulacoes longas (> 100 periodos).

---

## Proximos Passos

Apos dominar os conceitos deste notebook, explore:

1. **Metodos Espectrais** - Resolucao de EDPs usando transformadas de Fourier,
   ideal para problemas com condicoes periodicas

2. **Metodo de Elementos Finitos (FEM)** - Discretizacao mais flexivel para
   geometrias complexas e condicoes de contorno mistas

3. **Metodo Monte Carlo** - Simulacoes estatisticas para mecanica estatistica,
   transporte de particulas e integracao numerica

4. **Computacao Paralela** - Uso de multiprocessing, MPI e GPU (CUDA/OpenCL)
   para acelerar simulacoes de grande escala

5. **Machine Learning em Fisica** - Physics-Informed Neural Networks (PINNs)
   para resolver EDPs e modelos substitutos

6. **Validacao e Verificacao (V&V)** - Metodologias rigorosas para garantir
   que simulacoes representam a fisica corretamente

---

*Notebook criado para o modulo `11_engfis_tecnico` - Engenharia Fisica.*
*Feito com Claude (Anthropic).*